In [0]:
entities = ["customers", "products", "reviews", "categories", "suppliers", "payments", "inventory", "shipping"]

storage_account_name = "stecommercedirty01"  # el tuyo real
container = "bronze"

dataframes = {}  # aquí guardaremos cada DataFrame, con el nombre de la entidad como clave

for entity_name in entities:
    path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/{entity_name}/"
    df = spark.read.option("multiLine", True).json(path)
    dataframes[entity_name] = df
    print(f"{entity_name}: {df.count()} filas cargadas")

In [0]:
dataframes["customers"].printSchema()

In [0]:
dataframes["customers"].select("country").distinct().display()

In [0]:
dataframes["customers"].filter(dataframes["customers"].country == "Germany").display()

In [0]:
from pyspark.sql.functions import upper, trim, col

df_customers = dataframes["customers"]

# Normalizamos formato: quitamos espacios sobrantes y pasamos todo a mayúsculas
df_customers = df_customers.withColumn("country_clean", upper(trim(col("country"))))

In [0]:
from pyspark.sql.functions import when

df_customers = df_customers.withColumn(
    "country_final",
    when(col("country_clean").isin("USA", "US", "U.S.A", "U.S.A.", "UNITED STATES"), "USA")
    .when(col("country_clean").isin("CANADA"), "Canada")
    .when(col("country_clean").isin("MEXICO"), "Mexico")
    .when(col("country_clean").isin("UK", "UNITED KINGDOM"), "UK")
    .when(
        (col("country_clean") == "GERMANY") &
        (col("city").isin("Portland", "San Francisco", "Los Angeles", "Miami", "Denver", "Dallas", "Houston", "Phoenix", "los angeles")),
        "UNKNOWN"
    )
    .when((col("country_clean").isNull()) | (col("country_clean") == ""), "UNKNOWN")
    .otherwise(col("country_clean"))
)


In [0]:
df_customers.select("country", "country_clean", "country_final").distinct().display()

In [0]:
df_customers.filter(col("country") == "").display()

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/guillerg0101@gmail.com/ecommerce-etl-pipeline")

from transformacion.limpieza_customers import limpiar_customers

df_customers_limpio = limpiar_customers(dataframes["customers"])
df_customers_limpio.select("country", "country_clean", "country_final").distinct().display()